# The Linear Probability Model: Theoretical Foundations and Structural Limitations

## 1. Introduction to the Linear Probability Model (LPM)

In predictive modeling and statistical inference, the nature of the dependent variable dictates the choice of the algorithmic framework. When the outcome of interest is continuous and unbounded, Ordinary Least Squares (OLS) regression is the standard tool. However, in numerous enterprise and scientific applications, the outcome is dichotomous: a customer defaults or repays, a patient survives or expires, a user clicks or ignores.

The Linear Probability Model (LPM) attempts to model the conditional probability P(Y=1|X) directly as a linear combination of the covariates. It treats the binary outcome as if it were a continuous variable, estimating the parameters via standard OLS minimization of the sum of squared residuals.

While it violates the Gauss-Markov assumptions regarding error distribution and homoskedasticity, it remains a critical baseline in econometrics and causal inference due to its computational efficiency, direct interpretability of coefficients as marginal effects, and compatibility with high-dimensional fixed effects.

In [ ]:
# Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import time
import warnings

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Configure standard plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

# Set global random seed for complete reproducibility
np.random.seed(42)

print("Environment initialized successfully. Random seed set to 42.")

## 2. Data Creation: Simulating a Binary Outcome

To evaluate the Linear Probability Model, we must first construct a dataset with a binary dependent variable.

We will simulate an enterprise scenario: Predicting Loan Default. 
We have two continuous features: Credit Score and Income (in thousands). 

We will generate the true underlying probability using a logistic (S-curve) function to reflect reality, but we will purposefully attempt to fit this reality using the linear LPM.

In [ ]:
# Synthetic Data Generation: Loan Default
n_samples = 2000

# Feature 1: Credit Score (Normally distributed around 650)
credit_score = np.random.normal(650, 100, n_samples)

# Feature 2: Income in thousands (Normally distributed around 50)
income = np.random.normal(50, 20, n_samples)

# True Data Generating Process (DGP) using a non-linear logit function
# Higher credit scores and higher income reduce the log-odds of default
z = 7.0 - 0.01 * credit_score - 0.05 * income
true_prob = 1 / (1 + np.exp(-z))

# Generate binary outcomes (0 = Paid, 1 = Default) based on the true probability
y_binary = np.random.binomial(1, true_prob)

# Assemble into a Pandas DataFrame
df_loans = pd.DataFrame({
    'Credit_Score': credit_score,
    'Income_K': income,
    'True_Probability': true_prob,
    'Default': y_binary
})

print(f"Generated dataset with {n_samples} observations.")
print("Sample of the dataset:")
print(df_loans.head())

## 3. Core Concept 1: The LPM Mathematical Formulation

Let Y_i be a binary random variable such that Y_i in {0, 1}. Let X_i be a K x 1 vector of explanatory variables.

The expected value of a Bernoulli random variable is its probability of success:
E[Y_i | X_i] = 1 * P(Y_i = 1 | X_i) + 0 * P(Y_i = 0 | X_i) = p_i

The Linear Probability Model specifies this conditional expectation as a simple linear function:
E[Y_i | X_i] = X_i^T * beta

Therefore, the population model is:
Y_i = X_i^T * beta + epsilon_i

In [ ]:
# Exploratory Data Analysis (EDA)
print("--- Descriptive Statistics ---")
print(df_loans[['Credit_Score', 'Income_K', 'Default']].describe().round(2))

default_rate = df_loans['Default'].mean() * 100
print(f"\nOverall Loan Default Rate: {default_rate:.2f}%")

# Visualize the marginal relationship between Credit Score and Default
plt.figure(figsize=(8, 5))
sns.regplot(x='Credit_Score', y='Default', data=df_loans, 
            logistic=False, scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.title('Naive Linear Regression line through Binary Data')
plt.xlabel('Credit Score')
plt.ylabel('Default (1=Yes, 0=No)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The red line above is the essence of the Linear Probability Model. It draws a straight line through the 0s and 1s.")

## 4. Fitting the Linear Probability Model (Standard OLS)

We will use the `statsmodels` library to fit a standard Ordinary Least Squares (OLS) model to our binary outcome.

The primary advantage of the LPM is interpretive: the coefficient beta_k is exactly the constant marginal effect. A 1-unit increase in X_k changes the probability of Y=1 by beta_k.

In [ ]:
# Define features (X) and target (y)
features = ['Credit_Score', 'Income_K']
X = sm.add_constant(df_loans[features]) # add intercept
y = df_loans['Default']

# Fit the standard OLS model
lpm_standard = sm.OLS(y, X).fit()

print("--- Linear Probability Model Summary (Standard OLS) ---")
print(lpm_standard.summary().tables[1])

beta_credit = lpm_standard.params['Credit_Score']
beta_income = lpm_standard.params['Income_K']

print("\n--- Marginal Effects Interpretation ---")
print(f"For every 1-point increase in Credit Score, the probability of default changes by {beta_credit:.5f} ({beta_credit*100:.3f} percentage points).")
print(f"For every $1K increase in Income, the probability of default changes by {beta_income:.5f} ({beta_income*100:.3f} percentage points).")

## 5. Core Concept 2: Structural Flaw 1 - Unbounded Predictions

The fundamental tension of the LPM lies in the geometric mismatch between a linear function and a bounded probability space.

Because a straight line has a constant slope and extends to infinity in both directions, it will inevitably cross the Y=1 boundary and predict values greater than 1, and cross the Y=0 boundary to predict values less than 0.

Since probabilities cannot exceed 1 or fall below 0, these predictions are mathematically non-sensical.

In [ ]:
# Generate predictions from our standard LPM
df_loans['LPM_Predicted_Prob'] = lpm_standard.predict(X)

# Identify out-of-bounds predictions
out_of_bounds_high = df_loans['LPM_Predicted_Prob'] > 1.0
out_of_bounds_low = df_loans['LPM_Predicted_Prob'] < 0.0

print("--- Unbounded Prediction Analysis ---")
print(f"Total Observations: {len(df_loans)}")
print(f"Predictions > 1.0 (Impossible): {out_of_bounds_high.sum()}")
print(f"Predictions < 0.0 (Impossible): {out_of_bounds_low.sum()}")
print(f"Total Invalid Predictions: {(out_of_bounds_high | out_of_bounds_low).sum()} ({(out_of_bounds_high | out_of_bounds_low).mean()*100:.2f}% of data)")

print("\nExample of an impossible prediction:")
extreme_case = df_loans[out_of_bounds_low].head(1)
print(extreme_case[['Credit_Score', 'Income_K', 'LPM_Predicted_Prob']])

### Visualizing the Boundary Breach

Let's visualize this structural failure by plotting the predicted probabilities against a synthetic 1D feature axis.

In [ ]:
# Create a 1D synthetic dataset mapping a generic feature X to binary Y
np.random.seed(99)
x_synthetic = np.random.uniform(-5, 5, 500)
true_p_synth = 1 / (1 + np.exp(-x_synthetic)) # True relationship is a sigmoid curve
y_synthetic = np.random.binomial(1, true_p_synth)

# Fit LPM to synthetic 1D data
X_synth_sm = sm.add_constant(x_synthetic)
lpm_synth = sm.OLS(y_synthetic, X_synth_sm).fit()
preds_synth = lpm_synth.predict(X_synth_sm)

plt.figure(figsize=(10, 6))
plt.scatter(x_synthetic, y_synthetic, color='gray', alpha=0.3, label='Observed Data (0 or 1)')
plt.plot(x_synthetic, preds_synth, color='red', linewidth=3, label='LPM Linear Fit')

# Highlight boundaries
plt.axhline(1.0, color='black', linestyle='--', label='Valid Upper Bound (1.0)')
plt.axhline(0.0, color='black', linestyle='--', label='Valid Lower Bound (0.0)')

# Fill out-of-bounds regions
sorted_indices = np.argsort(x_synthetic)
x_sorted = x_synthetic[sorted_indices]
preds_sorted = preds_synth[sorted_indices]

plt.fill_between(x_sorted, 1.0, preds_sorted, where=(preds_sorted > 1.0), color='red', alpha=0.3, label='Impossible Region (>1)')
plt.fill_between(x_sorted, 0.0, preds_sorted, where=(preds_sorted < 0.0), color='red', alpha=0.3, label='Impossible Region (<0)')

plt.title('Flaw 1: The Boundary Breach in the Linear Probability Model', fontsize=14)
plt.xlabel('Generic Feature X', fontsize=12)
plt.ylabel('Predicted Probability P(Y=1|X)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Core Concept 3: Structural Flaw 2 - Heteroskedasticity

Because Y_i can only take the values 0 or 1, the error term epsilon_i = Y_i - X_i^T * beta is strictly limited to two possible outcomes for any given X_i.

The variance of the error term conditional on X_i is:
Var(epsilon_i | X_i) = p_i * (1 - p_i)

Because p_i = X_i^T * beta, the variance of the error term depends entirely on the X covariates. This is the exact definition of heteroskedasticity. The standard OLS assumption of constant variance is fundamentally violated.

In [ ]:
# Calculate the theoretical variance of the error term based on LPM predictions
# We must clip probabilities to [0.001, 0.999] so that variance doesn't become negative due to unbounded predictions
clipped_preds = np.clip(df_loans['LPM_Predicted_Prob'], 0.001, 0.999)
theoretical_variance = clipped_preds * (1 - clipped_preds)

# Calculate actual empirical residuals squared
df_loans['LPM_Residuals'] = lpm_standard.resid
df_loans['Squared_Residuals'] = df_loans['LPM_Residuals'] ** 2

print("--- Analyzing Heteroskedasticity ---")
print("Notice how theoretical variance peaks exactly at P(Y=1|X) = 0.5 and approaches 0 at the boundaries.")

sample_variances = pd.DataFrame({
    'Predicted_Prob': clipped_preds,
    'Theoretical_Variance': theoretical_variance
}).sort_values('Predicted_Prob').iloc[::200] # take a sample across the distribution

print(sample_variances.to_string(index=False))

### Visualizing Heteroskedasticity: The Fan Shape

If we plot the residuals against the predicted values, the variance is not a uniform cloud. It forms a distinct fan or inverted-U shape. The variance reaches its maximum at p=0.5.

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot of Predicted Probabilities vs Residuals
plt.scatter(df_loans['LPM_Predicted_Prob'], df_loans['LPM_Residuals'], 
            alpha=0.4, color='purple', label='Actual Residuals')

plt.axhline(0, color='black', linestyle='-', linewidth=2)

plt.title('Flaw 2: Heteroskedasticity in the LPM (The Fan Shape)', fontsize=14)
plt.xlabel('LPM Predicted Probability P(Y=1|X)', fontsize=12)
plt.ylabel('Residual (Y - Predicted)', fontsize=12)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("The two distinct lines of points correspond to actual Y=0 and actual Y=1.")
print("The spread (variance) is clearly not constant across the x-axis.")

## 7. Core Concept 4: The Mandatory Fix - Robust Standard Errors

Because the LPM is inherently heteroskedastic, the default standard errors produced by standard OLS are biased and inconsistent. Any hypothesis tests (p-values, confidence intervals) derived from them will be invalid.

We must compute heteroskedasticity-consistent covariance matrices, typically HC2 or HC3 (White's Robust Standard Errors), to perform valid statistical inference.

In [ ]:
# Fit the LPM using Heteroskedasticity-Robust Standard Errors (HC3)
lpm_robust = sm.OLS(y, X).fit(cov_type='HC3')

# Compare the standard errors side-by-side
comparison_df = pd.DataFrame({
    'Coefficient': lpm_standard.params,
    'Standard SE (Biased)': lpm_standard.bse,
    'Robust SE (HC3)': lpm_robust.bse,
    'Standard P-val': lpm_standard.pvalues,
    'Robust P-val': lpm_robust.pvalues
})

print("--- Standard Errors Comparison ---")
print("Always use robust SEs when estimating a Linear Probability Model.")
print(comparison_df.round(6))

# Calculate percentage difference in standard errors
se_diff_pct = ((comparison_df['Robust SE (HC3)'] - comparison_df['Standard SE (Biased)']) / comparison_df['Standard SE (Biased)']) * 100
print("\n--- Percentage Change in Standard Errors ---")
print(se_diff_pct.round(2).astype(str) + "%")

## 8. Real-World Applications: Why use the LPM?

If the LPM produces impossible probabilities and has heteroskedastic residuals, why is it still used heavily in causal inference and econometrics?

1. High-Dimensional Fixed Effects: In panel datasets with millions of individuals and time periods, estimating non-linear models (like Logit) with fixed effects is computationally intractable and suffers from the 'incidental parameters problem' (estimates are inconsistent).
2. Computational Speed: Matrix inversion (X^T X)^-1 X^T Y takes a fraction of a second, whereas Maximum Likelihood Estimation (MLE) requires heavy iterative algorithms.
3. Interpretable Marginal Effects: The coefficient is the average marginal effect directly, requiring no post-estimation math.

In [ ]:
# Simulating computational speed difference between LPM (OLS) and Logit (MLE)
# We will use a larger dataset to emphasize the difference
n_large = 100000
X_large = np.random.normal(0, 1, (n_large, 10))
X_large_sm = sm.add_constant(X_large)

# Create binary target
true_z_large = np.sum(X_large * np.random.uniform(-0.5, 0.5, 10), axis=1)
true_p_large = 1 / (1 + np.exp(-true_z_large))
y_large = np.random.binomial(1, true_p_large)

print(f"Benchmarking on dataset with {n_large} rows and 10 features...")

# Time LPM (OLS)
start_time = time.time()
lpm_large = sm.OLS(y_large, X_large_sm).fit(cov_type='HC3')
lpm_time = time.time() - start_time
print(f"LPM (OLS) execution time: {lpm_time:.4f} seconds")

# Time Logit (MLE)
start_time = time.time()
logit_large = sm.Logit(y_large, X_large_sm).fit(disp=0)
logit_time = time.time() - start_time
print(f"Logit (MLE) execution time: {logit_time:.4f} seconds")

print(f"\nThe LPM is approximately {logit_time / lpm_time:.1f}x faster.")
print("In big data environments with thousands of fixed effects (dummy variables), Logit often fails to converge entirely.")

## 9. Practice Exercise: A/B Testing with LPM

In A/B testing, the independent variable is a binary treatment flag (0 or 1), and the dependent variable is a binary conversion flag (0 or 1). Because both variables are binary, the LPM boundary issue does not apply, making it a perfect tool.

Exercise: Calculate the causal impact of the treatment on the conversion rate using the LPM with robust standard errors.

In [ ]:
# Exercise Setup: Synthetic A/B Testing Data
np.random.seed(101)
n_users = 5000

# 50/50 split between control (0) and treatment (1)
treatment_group = np.random.binomial(1, 0.5, n_users)

# Control converts at ~5%, Treatment converts at ~8%
base_conv_rate = 0.05
treatment_lift = 0.03
conversion_probs = base_conv_rate + (treatment_group * treatment_lift)

converted = np.random.binomial(1, conversion_probs)

df_ab = pd.DataFrame({'Treatment': treatment_group, 'Converted': converted})
print("A/B Test Data Ready.")
print(df_ab['Converted'].value_counts())

### Exercise Solution

In [ ]:
# Solution Code
# 1. Define X and y
X_ab = sm.add_constant(df_ab['Treatment'])
y_ab = df_ab['Converted']

# 2. Fit LPM with HC3 robust standard errors
lpm_ab_test = sm.OLS(y_ab, X_ab).fit(cov_type='HC3')

# 3. Extract and interpret results
baseline_conversion = lpm_ab_test.params['const']
treatment_effect = lpm_ab_test.params['Treatment']
p_value = lpm_ab_test.pvalues['Treatment']

print("--- A/B Test Causal Impact Report (via LPM) ---")
print(f"Baseline Conversion Rate (Control): {baseline_conversion*100:.2f}%")
print(f"Absolute Lift (Treatment Effect):   {treatment_effect*100:.2f} percentage points")
print(f"P-Value (Robust):                   {p_value:.5f}")

if p_value < 0.05:
    print("\nConclusion: The treatment resulted in a statistically significant increase in conversion.")
else:
    print("\nConclusion: The treatment did not result in a statistically significant change.")

## 10. Visualization Gallery: LPM vs Logistic Regression

To conclude, let's contrast the structural form of the LPM against its non-linear counterpart, Logistic Regression (Logit). 

The Logit model utilizes the S-shaped sigmoid function to compress the linear prediction safely into the [0, 1] bounds, elegantly solving the boundary breach problem.

In [ ]:
# Re-using the synthetic 1D data from earlier
# Fit Logit model for comparison
logit_synth = sm.Logit(y_synthetic, X_synth_sm).fit(disp=0)
logit_preds = logit_synth.predict(X_synth_sm)

plt.figure(figsize=(12, 7))
plt.scatter(x_synthetic, y_synthetic, color='gray', alpha=0.3, label='Observed Data (0 or 1)')

# Plot the straight line from LPM
plt.plot(x_sorted, preds_sorted, color='red', linewidth=3, linestyle='--', label='LPM (Constant Marginal Effect)')

# Plot the S-curve from Logit
logit_sorted_preds = logit_preds[sorted_indices]
plt.plot(x_sorted, logit_sorted_preds, color='blue', linewidth=3, label='Logit (S-Curve, Diminishing Returns)')

plt.axhline(1.0, color='black', linestyle=':', alpha=0.6)
plt.axhline(0.0, color='black', linestyle=':', alpha=0.6)

plt.title('Comparison: Linear Probability Model vs Logistic Regression', fontsize=16)
plt.xlabel('Generic Feature X', fontsize=12)
plt.ylabel('Predicted Probability', fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

### Explaining the Visual Differences

The blue Logit line exhibits "Diminishing Returns". Near the middle (P=0.5), a small change in X causes a large change in probability. Near the extremes (P=0.99), a change in X causes almost no change in probability.

The red LPM line exhibits a "Constant Marginal Effect". Every unit of X adds the exact same amount of probability, regardless of where you are on the spectrum, eventually driving the model through the floor and ceiling.

## 11. Summary and Key Takeaways

- **The Linear Probability Model** applies standard OLS regression to a binary outcome, modeling P(Y=1|X) = X^T * beta.
- **Fatal Flaws**: It systematically produces unbounded predictions (outside [0, 1]) and exhibits severe heteroskedasticity, violating core Gauss-Markov assumptions.
- **Mandatory Correction**: Default OLS standard errors are invalid. Heteroskedasticity-robust standard errors (HC2/HC3) must be used for all hypothesis testing and inference.
- **Primary Advantage**: The coefficients are directly interpretable as constant marginal effects (percentage point changes), and the model is computationally trivial to estimate.
- **Engineering Utility**: Remains heavily utilized in panel data econometrics for causal inference, Instrumental Variables (2SLS), and high-dimensional fixed-effect models where non-linear MLE estimators fail or become statistically inconsistent.